In [2]:
import os
import warnings
from pathlib import Path

import librosa
import torch
from IPython.display import Audio
from dotenv import load_dotenv
from pyannote.audio import Pipeline

load_dotenv()

/home/dom/GitRepos/multilingual-transcription/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
SAMPLE_RATE = 16000

In [4]:
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", token=os.environ["HUGGINGFACE_TOKEN"])

In [5]:
def load_audio_as_mapping(audio_path: str|Path) -> dict:
    waveform, samplerate = librosa.load(audio_path, sr=SAMPLE_RATE)

    waveform = torch.from_numpy(waveform).unsqueeze(0).float()

    # Compatible with pyannote, and allows us to easily split waveform into smaller chunks
    audio_mapping = {
        "waveform": waveform,
        "sample_rate": samplerate,
        "channel": 0,
        "uri": audio_path.name,
    }
    return audio_mapping

In [12]:
audio_path = Path("../data/example_2/audio.mp3")
audio = load_audio_as_mapping(audio_path)

In [13]:
with warnings.catch_warnings(action="ignore"):
    diary = pipeline(audio)

In [14]:
annotation = diary.speaker_diarization

for turn, _, speaker in annotation.itertracks(yield_label=True):
    print(f"[{turn.start:.2f}s - {turn.end:.2f}s] {speaker}")

[0.03s - 0.57s] SPEAKER_00
[0.74s - 1.47s] SPEAKER_00
[1.90s - 2.83s] SPEAKER_00
[3.51s - 5.18s] SPEAKER_01
[5.58s - 7.14s] SPEAKER_01
[7.83s - 8.32s] SPEAKER_00
[8.70s - 10.27s] SPEAKER_00
[10.97s - 11.54s] SPEAKER_01
[11.94s - 13.28s] SPEAKER_01
[13.89s - 15.03s] SPEAKER_00
[15.32s - 16.84s] SPEAKER_00
[17.58s - 20.23s] SPEAKER_01
[20.57s - 21.66s] SPEAKER_01
[22.46s - 23.96s] SPEAKER_00
[24.40s - 26.04s] SPEAKER_00
[26.74s - 28.14s] SPEAKER_01
[28.74s - 30.73s] SPEAKER_01
[31.64s - 32.16s] SPEAKER_00
[32.48s - 34.42s] SPEAKER_00
[35.28s - 38.35s] SPEAKER_01
[39.10s - 40.28s] SPEAKER_00
[40.55s - 41.51s] SPEAKER_00
[42.54s - 43.91s] SPEAKER_01
[44.73s - 45.34s] SPEAKER_00
[45.46s - 46.17s] SPEAKER_00
[47.06s - 48.16s] SPEAKER_01
[48.55s - 50.44s] SPEAKER_01
[51.38s - 53.10s] SPEAKER_00
[53.79s - 56.55s] SPEAKER_01
[57.44s - 58.06s] SPEAKER_00
[58.17s - 59.11s] SPEAKER_00


In [41]:
# Slicing the audio to individual speaker segments
segments = []
previous_speaker = None

for turn, _, speaker in annotation.itertracks(yield_label=True):
    start_idx = int(turn.start * SAMPLE_RATE)
    end_idx = int(turn.end * SAMPLE_RATE)
    segment = audio["waveform"][0, start_idx:end_idx]

    if speaker != previous_speaker:
        # another speaker is now
        segments.append(segment)
    else:
        # The same speaker keeps talking
        previous_segment = segments[-1]
        merged_segments = torch.concat([previous_segment, segment], dim=0)
        segments[-1] = merged_segments

    previous_speaker = speaker

In [42]:
segments

[tensor([-0.0004, -0.0004, -0.0004,  ...,  0.0009,  0.0007, -0.0008]),
 tensor([-0.0058, -0.0080, -0.0047,  ..., -0.0002,  0.0060,  0.0140]),
 tensor([ 0.0013, -0.0019, -0.0009,  ..., -0.0004, -0.0033, -0.0059]),
 tensor([0.0026, 0.0032, 0.0030,  ..., 0.0818, 0.0854, 0.0885]),
 tensor([-0.0002, -0.0002, -0.0002,  ...,  0.0164,  0.0167,  0.0169]),
 tensor([ 0.0017,  0.0187, -0.0299,  ..., -0.0412, -0.0342, -0.0274]),
 tensor([ 3.7621e-05,  4.5497e-05,  5.2450e-05,  ..., -1.3810e-02,
         -1.4708e-02, -1.0016e-02]),
 tensor([ 2.8329e-05,  2.9139e-05,  5.5946e-05,  ..., -2.7816e-03,
         -4.6679e-03, -7.3381e-03]),
 tensor([-0.0001, -0.0001, -0.0001,  ..., -0.0138, -0.0130, -0.0067]),
 tensor([ 1.8870e-05,  2.4681e-05,  2.2755e-05,  ..., -9.5132e-03,
         -1.1614e-02, -1.3119e-02]),
 tensor([-0.0001, -0.0001, -0.0001,  ...,  0.0006, -0.0045,  0.0106]),
 tensor([0.0264, 0.0120, 0.0109,  ..., 0.0168, 0.0242, 0.0252]),
 tensor([ 2.0533e-05,  1.5977e-05,  2.6812e-05,  ...,  1.1200

In [43]:
Audio(segments[0], rate=SAMPLE_RATE)

In [44]:
Audio(segments[1], rate=SAMPLE_RATE)

In [45]:
Audio(segments[2], rate=SAMPLE_RATE)

In [46]:
Audio(segments[3], rate=SAMPLE_RATE)

In [47]:
Audio(segments[4], rate=SAMPLE_RATE)